In [1]:

import csv
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

DURATION = 480.0
REPLICAS = 30
SEED = 20261122
TYPES = ('Rápido', 'Normal', 'Lento', 'Muy lento')
DATA = {
 'Retiro': [(1,1,.23),(2,2,.40),(3,3,.17),(4,3,.20)],
 'Pago': [(3,1,.10),(3,2,.20),(5,3,.30),(7,4,.40)],
}
SCENARIOS = {
 '3 mixtos': ('Mixto','Mixto','Mixto'),
 '1 retiro + 2 pagos': ('Retiro','Pago','Pago'),
 '2 retiros + 1 pago': ('Retiro','Retiro','Pago'),
 '4 cajeros (2+2)': ('Retiro','Retiro','Pago','Pago'),
}

def weighted_pick(rng, weights):
    x = rng.random()
    s = 0.0
    for i, w in enumerate(weights):
        s += w
        if x < s:
            return i
    return len(weights)-1

def generate_users(seed):
    rng = random.Random(seed)
    arrivals = []
    clock = 0.0
    while True:
        action = 'Retiro' if rng.random() < .70 else 'Pago'
        k = weighted_pick(rng, [x[2] for x in DATA[action]])
        service_mean, arrival_mean, _ = DATA[action][k]
        clock += rng.expovariate(1 / arrival_mean)
        if clock > DURATION:
            break
        service = rng.expovariate(1 / service_mean)
        arrivals.append((clock, action, TYPES[k], service))
    return arrivals

def simulate(users, assignment, rep):
    free_at = [0.0] * len(assignment)
    all_rows = []
    for arrive, action, user_type, service in users:
        eligible = [i for i, role in enumerate(assignment) if role in ('Mixto',action)]

        cashier = min(eligible, key=lambda i: (max(0,free_at[i]-arrive),i))
        start = max(arrive,free_at[cashier])
        finish = start + service
        free_at[cashier] = finish
        all_rows.append({'replica':rep,'cajero':cashier+1,'accion':action,
            'tipo':user_type,'llegada':arrive,'inicio':start,'fin':finish,
            'espera':start-arrive,'servicio':service,'sistema':finish-arrive,
            'terminado':int(finish <= DURATION)})
    return all_rows

def avg(values):
    return sum(values)/len(values) if values else 0.0

def aggregate(rows, assignment, replica):
    active = [r for r in rows if r['terminado']]
    by_cash = []
    for i in range(1,len(assignment)+1):
        subset=[r for r in rows if r['cajero']==i]
        completed=[r for r in subset if r['terminado']]
        busy = sum(max(0.,min(DURATION,r['fin'])-max(0.,r['inicio'])) for r in subset)
        by_cash.append({'replica':replica,'cajero':i,'rol':assignment[i-1],
            'usuarios_asignados':len(subset),'terminados':len(completed),
            'espera_media':avg([r['espera'] for r in completed]),
            'atencion_media':avg([r['servicio'] for r in completed]),
            'sistema_medio':avg([r['sistema'] for r in completed]),
            'ocupacion':busy/DURATION})
    totals=Counter((r['accion'],r['tipo']) for r in rows)
    row={'replica':replica,'llegadas':len(rows),'terminados':len(active),
         'pendientes':len(rows)-len(active),
         'espera_media':avg([r['espera'] for r in active]),
         'sistema_medio':avg([r['sistema'] for r in active]),
         'atencion_media':avg([r['servicio'] for r in active]),
         'espera_retiro':avg([r['espera'] for r in active if r['accion']=='Retiro']),
         'espera_pago':avg([r['espera'] for r in active if r['accion']=='Pago']),
         'espera_maxima':max([r['espera'] for r in active],default=0),
         'pct_espera_mayor_5':sum(r['espera']>5 for r in active)/len(active) if active else 0}
    for action in DATA:
        for typ in TYPES:
            row[f'{action}_{typ}']=totals[(action,typ)]
    return row,by_cash

def run():

    outdir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    all_summary=[]; all_cash=[]; all_people=[]
    for rep in range(1,REPLICAS+1):
        users=generate_users(SEED + rep)
        for scenario, assignment in SCENARIOS.items():
            rows=simulate(users,assignment,rep)
            summary,cash=aggregate(rows,assignment,rep)
            summary['escenario']=scenario
            all_summary.append(summary)
            for c in cash:
                c['escenario']=scenario
                all_cash.append(c)
            for r in rows:
                r['escenario']=scenario
                all_people.append(r)
    for name, dataset in [('resultados_replicas.csv',all_summary),
                          ('resultados_cajeros.csv',all_cash),
                          ('resultados_usuarios.csv',all_people)]:
        with (outdir/name).open('w',newline='',encoding='utf-8-sig') as f:
            writer=csv.DictWriter(f,fieldnames=list(dataset[0]))
            writer.writeheader();writer.writerows(dataset)
    print('MEDIA DE LAS 30 RÉPLICAS')
    for scenario in SCENARIOS:
        group=[x for x in all_summary if x['escenario']==scenario]
        print(scenario,{k:round(avg([x[k] for x in group]),3) for k in
            ['llegadas','terminados','pendientes','espera_media','espera_retiro','espera_pago','sistema_medio','pct_espera_mayor_5']})
        for i in range(1,len(SCENARIOS[scenario])+1):
            subset=[x for x in all_cash if x['escenario']==scenario and x['cajero']==i]
            print(' caja',i,subset[0]['rol'],{k:round(avg([x[k] for x in subset]),3) for k in ['atencion_media','espera_media','ocupacion','terminados']})
    print('Clases por replica (iguales en todos los escenarios):')
    ss=[x for x in all_summary if x['escenario']=='3 mixtos']
    for action in DATA:
        for typ in TYPES:
            key=f'{action}_{typ}'
            print(key,'promedio',round(avg([x[key] for x in ss]),2),'minimo',min((x[key],x['replica']) for x in ss))
    print('min llegadas',min((x['llegadas'],x['replica']) for x in ss))
    return all_summary, all_cash

if __name__=='__main__':
    run()


MEDIA DE LAS 30 RÉPLICAS
3 mixtos {'llegadas': 203.933, 'terminados': 202.133, 'pendientes': 1.8, 'espera_media': 0.472, 'espera_retiro': 0.496, 'espera_pago': 0.417, 'sistema_medio': 3.67, 'pct_espera_mayor_5': 0.025}
 caja 1 Mixto {'atencion_media': 3.496, 'espera_media': 0.354, 'ocupacion': 0.616, 'terminados': 84.267}
 caja 2 Mixto {'atencion_media': 3.036, 'espera_media': 0.445, 'ocupacion': 0.444, 'terminados': 69.667}
 caja 3 Mixto {'atencion_media': 2.987, 'espera_media': 0.699, 'ocupacion': 0.305, 'terminados': 48.2}
1 retiro + 2 pagos {'llegadas': 203.933, 'terminados': 200.333, 'pendientes': 3.6, 'espera_media': 5.346, 'espera_retiro': 7.369, 'espera_pago': 0.566, 'sistema_medio': 8.555, 'pct_espera_mayor_5': 0.327}
 caja 1 Retiro {'atencion_media': 2.403, 'espera_media': 7.369, 'ocupacion': 0.71, 'terminados': 140.967}
 caja 2 Pago {'atencion_media': 5.201, 'espera_media': 0.428, 'ocupacion': 0.429, 'terminados': 39.4}
 caja 3 Pago {'atencion_media': 5.092, 'espera_media': 